# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/titlyzaman25/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
import os
import random
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.inspection import permutation_importance

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)

Seed: 42


In [26]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(march_file)

print("March performance rows:", len(march))
print("March date range:", march["report_date"].min(), "to", march["report_date"].max())

March performance rows: 9841378
March date range: 2026-03-01 to 2026-03-31


In [27]:
content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content = pd.read_parquet(content_file)

print("Content rows:", len(content))
print("Content columns:", content.columns.tolist())

Content rows: 519606
Content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [28]:
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april = pd.read_parquet(april_file)

print("April performance rows:", len(april))
print(
    "April date range:",
    april["report_date"].min(),
    "to",
    april["report_date"].max()
)

April performance rows: 10424730
April date range: 2026-04-01 to 2026-04-30


In [29]:
march_content = (
    march
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum")
    )
)

print("March content items:", len(march_content))
print(
    "Unique content items:",
    march_content["content_hash_id"].nunique()
)

March content items: 331437
Unique content items: 331437


In [30]:
march_content["ctr"] = np.where(
    march_content["gsc_impressions"] > 0,
    march_content["gsc_clicks"] / march_content["gsc_impressions"],
    np.nan
)

print("Valid CTR:", march_content["ctr"].notna().sum())

Valid CTR: 176738


In [31]:
metadata_cols = [
    "client_hash_id",
    "content_hash_id",
    "content_created_date",
    "content_updated_date",
    "last_optimized_date",
    "content_type",
    "search_volume",
    "competition",
    "main_intent",
    "word_count",
    "char_count",
    "is_published",
    "is_deleted"
]

content_meta = content[metadata_cols].copy()

baseline = march_content.merge(
    content_meta,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("After metadata merge:", len(baseline))
print(
    "Unique content items:",
    baseline["content_hash_id"].nunique()
)

After metadata merge: 331437
Unique content items: 331437


In [32]:
CUTOFF_DATE = pd.Timestamp("2026-03-31")

baseline["content_updated_date"] = pd.to_datetime(
    baseline["content_updated_date"],
    errors="coerce"
)

baseline["days_since_update"] = (
    CUTOFF_DATE - baseline["content_updated_date"]
).dt.days

# Remove future update dates from the feature
baseline.loc[
    baseline["days_since_update"] < 0,
    "days_since_update"
] = np.nan

print("Future update dates excluded:", baseline["days_since_update"].isna().sum())
print(baseline["days_since_update"].describe())

Future update dates excluded: 293358
count    38079.000000
mean        65.360592
std         67.131240
min          7.000000
25%         34.000000
50%         34.000000
75%         34.000000
max        303.000000
Name: days_since_update, dtype: float64


In [33]:
april_content = (
    april
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
)

print("April content outcomes:", len(april_content))
april_content["future_click_label"] = (
    april_content["april_clicks"] > 0
).astype(int)

print(
    april_content["future_click_label"].value_counts()
)

print(
    "Future label rate:",
    april_content["future_click_label"].mean()
)

April content outcomes: 362172
future_click_label
0    294340
1     67832
Name: count, dtype: int64
Future label rate: 0.18729222579326948


In [34]:
model_df = baseline.merge(
    april_content[
        [
            "client_hash_id",
            "content_hash_id",
            "future_click_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Final modeling rows:", len(model_df))
print(
    "Label rate:",
    model_df["future_click_label"].mean()
)

Final modeling rows: 331436
Label rate: 0.18736045571392365


In [35]:
FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_total_engagement_sec",
    "ctr",
    "days_since_update"
]

print("Candidate features:")
for feature in FEATURES:
    print("-", feature)

print("\nMissing values:")
print(model_df[FEATURES].isna().sum())

Candidate features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_total_engagement_sec
- ctr
- days_since_update

Missing values:
gsc_impressions                  0
gsc_clicks                       0
gsc_avg_position            154699
ga4_sessions                     0
ga4_total_engagement_sec         0
ctr                         154699
days_since_update           293357
dtype: int64


In [36]:
print("Method: Logistic Regression")
print("Task: binary prediction of future April clicks")
print("Features:", FEATURES)
print("Base rate:", round(model_df["future_click_label"].mean(), 4))

Method: Logistic Regression
Task: binary prediction of future April clicks
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'ctr', 'days_since_update']
Base rate: 0.1874


In [37]:
X = model_df[FEATURES].copy()
y = model_df["future_click_label"].copy()
groups = model_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train clients:",
    model_df.iloc[train_idx]["client_hash_id"].nunique()
)

print(
    "Test clients:",
    model_df.iloc[test_idx]["client_hash_id"].nunique()
)

overlap = set(
    model_df.iloc[train_idx]["client_hash_id"]
) & set(
    model_df.iloc[test_idx]["client_hash_id"]
)

print("Client overlap:", len(overlap))

print("Train label rate:", y_train.mean())
print("Test label rate:", y_test.mean())

Train rows: 300879
Test rows: 30557
Train clients: 44
Test clients: 11
Client overlap: 0
Train label rate: 0.1848849537521728
Test label rate: 0.21173544523349805


In [38]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED))
])

model.fit(X_train, y_train)

print("Model trained successfully.")
model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.5).astype(int)

print("Predictions generated.")
model_auc = roc_auc_score(y_test, model_prob)
model_precision = precision_score(y_test, model_pred, zero_division=0)
model_recall = recall_score(y_test, model_pred, zero_division=0)
model_f1 = f1_score(y_test, model_pred, zero_division=0)
model_accuracy = accuracy_score(y_test, model_pred)

print("Model metrics")
print("----------------")
print("Accuracy :", round(model_accuracy, 4))
print("Precision:", round(model_precision, 4))
print("Recall   :", round(model_recall, 4))
print("F1       :", round(model_f1, 4))
print("ROC-AUC  :", round(model_auc, 4))

Model trained successfully.
Predictions generated.
Model metrics
----------------
Accuracy : 0.8882
Precision: 0.9311
Recall   : 0.5097
F1       : 0.6588
ROC-AUC  : 0.92


In [39]:
baseline_df = model_df.copy()

baseline_df["stale"] = (baseline_df["days_since_update"] >= 181).astype(int)

baseline_df["ctr"] = np.where(
    baseline_df["gsc_impressions"] > 0,
    baseline_df["gsc_clicks"] / baseline_df["gsc_impressions"],
    np.nan
)

baseline_df["position_group"] = np.nan
baseline_df.loc[baseline_df["gsc_avg_position"].between(1, 3), "position_group"] = "1-3"
baseline_df.loc[baseline_df["gsc_avg_position"].between(4, 10), "position_group"] = "4-10"
baseline_df.loc[baseline_df["gsc_avg_position"].between(11, 20), "position_group"] = "11-20"
baseline_df.loc[baseline_df["gsc_avg_position"] > 20, "position_group"] = "21+"

train_baseline = baseline_df.iloc[train_idx].copy()

group_expected_ctr = (
    train_baseline
    .dropna(subset=["position_group"])
    .groupby("position_group")["ctr"]
    .mean()
)

print("Expected CTR by position group:")
print(group_expected_ctr)

baseline_df["expected_ctr"] = baseline_df["position_group"].map(group_expected_ctr)
baseline_df["expected_ctr"] = pd.to_numeric(baseline_df["expected_ctr"], errors="coerce")
baseline_df["ctr"] = pd.to_numeric(baseline_df["ctr"], errors="coerce")

baseline_df["low_ctr"] = (
    (baseline_df["ctr"] < baseline_df["expected_ctr"])
    & baseline_df["ctr"].notna()
    & baseline_df["expected_ctr"].notna()
)

baseline_df["baseline_score"] = (
    baseline_df["stale"].astype(int) + baseline_df["low_ctr"].astype(int)
)

baseline_df["baseline_pred"] = (baseline_df["baseline_score"] > 0).astype(int)

print("\nBaseline score distribution:")
print(baseline_df["baseline_score"].value_counts().sort_index())
print("\nLow CTR count:", baseline_df["low_ctr"].sum())
print("Stale count:", baseline_df["stale"].sum())

/tmp/ipykernel_2171/353728757.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1-3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  baseline_df.loc[baseline_df["gsc_avg_position"].between(1, 3), "position_group"] = "1-3"


Expected CTR by position group:
position_group
1-3      0.011034
11-20    0.003410
21+      0.002017
4-10     0.004894
Name: ctr, dtype: float64

Baseline score distribution:
baseline_score
0    191478
1    139726
2       232
Name: count, dtype: int64

Low CTR count: 136374
Stale count: 3816


In [40]:
print("Position groups:")
print(baseline_df["position_group"].value_counts(dropna=False))

print("\nExpected CTR summary:")
print(baseline_df["expected_ctr"].describe())

print("\nBaseline prediction distribution:")
print(baseline_df["baseline_pred"].value_counts())

Position groups:
position_group
NaN      172917
4-10      70957
21+       44969
11-20     27488
1-3       15105
Name: count, dtype: int64

Expected CTR summary:
count    158519.000000
mean          0.004406
std           0.002466
min           0.002017
25%           0.002017
50%           0.004894
75%           0.004894
max           0.011034
Name: expected_ctr, dtype: float64

Baseline prediction distribution:
baseline_pred
0    191478
1    139958
Name: count, dtype: int64


In [41]:
baseline_test = baseline_df.iloc[test_idx].copy()

baseline_pred_test = baseline_test["baseline_pred"].values
baseline_score_test = baseline_test["baseline_score"].values

baseline_precision = precision_score(y_test, baseline_pred_test, zero_division=0)
baseline_recall = recall_score(y_test, baseline_pred_test, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_pred_test, zero_division=0)
baseline_auc = roc_auc_score(y_test, baseline_score_test)

print("WEEK-4 BASELINE — TEST SET")
print("=" * 40)
print("Precision:", round(baseline_precision, 4))
print("Recall   :", round(baseline_recall, 4))
print("F1       :", round(baseline_f1, 4))
print("ROC-AUC  :", round(baseline_auc, 4))
print("Base rate:", round(y_test.mean(), 4))

WEEK-4 BASELINE — TEST SET
Precision: 0.295
Recall   : 0.6485
F1       : 0.4056
ROC-AUC  : 0.6144
Base rate: 0.2117


In [42]:
comparison = pd.DataFrame({
    "method": ["Week-4 rule baseline", "Logistic Regression"],
    "precision": [baseline_precision, model_precision],
    "recall": [baseline_recall, model_recall],
    "f1": [baseline_f1, model_f1],
    "roc_auc": [baseline_auc, model_auc],
    "base_rate": [y_test.mean(), y_test.mean()]
})

print(comparison.round(4).to_string(index=False))

              method  precision  recall     f1  roc_auc  base_rate
Week-4 rule baseline     0.2950  0.6485 0.4056   0.6144     0.2117
 Logistic Regression     0.9311  0.5097 0.6588   0.9200     0.2117


In [43]:
lr_model = model.named_steps["classifier"]
feature_names = X_train.columns

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lr_model.coef_[0]
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

print("TOP 10 FEATURES BY ABSOLUTE COEFFICIENT")
display(coef_df.sort_values("abs_coefficient", ascending=False).head(10)[["feature", "coefficient"]])

test_pred = model.predict(X_test)
test_prob = model.predict_proba(X_test)[:, 1]

error_df = X_test.copy().reset_index(drop=True)
y_test_reset = pd.Series(y_test).reset_index(drop=True)
error_df["actual"] = y_test_reset
error_df["predicted"] = test_pred
error_df["probability"] = test_prob

false_positive = error_df[(error_df["actual"] == 0) & (error_df["predicted"] == 1)].copy()
false_negative = error_df[(error_df["actual"] == 1) & (error_df["predicted"] == 0)].copy()

print("\nERROR COUNTS")
print("False positives:", len(false_positive))
print("False negatives:", len(false_negative))

print("\nTHREE FALSE POSITIVES")
display(false_positive.head(3))

print("\nTHREE FALSE NEGATIVES")
display(false_negative.head(3))

TOP 10 FEATURES BY ABSOLUTE COEFFICIENT


,feature,coefficient
1,gsc_clicks,11.595015
0,gsc_impressions,2.177006
6,days_since_update,-0.378985
3,ga4_sessions,0.088232
2,gsc_avg_position,-0.087279
5,ctr,-0.016977
4,ga4_total_engagement_sec,0.014325



ERROR COUNTS
False positives: 244
False negatives: 3172

THREE FALSE POSITIVES


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_total_engagement_sec,ctr,days_since_update,actual,predicted,probability
20,3422,4,5.609351,13.0,372.0,0.001169,NaN,0,1,0.900605
171,2108,2,4.691132,1.0,0.0,0.000949,NaN,0,1,0.503248
729,4585,0,5.121635,14.0,0.0,0.000000,NaN,0,1,0.517979



THREE FALSE NEGATIVES


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_total_engagement_sec,ctr,days_since_update,actual,predicted,probability
6,0,0,NaN,0.0,0.0,NaN,NaN,1,0,0.071187
32,94,0,7.200812,2.0,1.0,0.000000,NaN,1,0,0.075813
34,107,1,10.567659,3.0,1.0,0.009346,NaN,1,0,0.138162


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
## 1. Two paper findings + my methodology questions

**Finding #4 — The Freshness Multiplier (361+ day growth ratio = 283:1)**
Methodology question: this bucket is reported at 283 growing vs. only 1 declining page.
The paper's own stated minimum sample size is 50 per bucket, and it flags this ratio as
unstable — but the 283:1 number still appears as a headline stat before that caveat.
Where does the label (growing/declining) come from? It's `trend_direction`, computed
from 30d-vs-prev-30d impression change — a short window that can be noisy for a bucket
this thin. Does the validation design (n=1 in the minority class) support treating this
as a "measured lever" rather than an artifact of small-sample variance?

**ML Appendix — Random Forest feature importance for Health Score**
Methodology question: Health Score is a composite built from Impressions (30 pts) +
Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). The Random Forest then predicts
Health Score using Impressions, Position, and CTR as top features. Since the label is
partly *constructed from* the inputs, is this importance ranking measuring anything
external, or mostly recovering the composite's own formula? The paper does flag this as
"descriptive, not causal" — the question is whether that caveat is prominent enough given
how the 43%/32%/15% importance numbers are presented.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [45]:
# ============================================
# 2. My model under an honest split (before/after)
# ============================================
# Week-5 used a GROUPED split by client_hash_id from the start.
# To show why that matters, re-run the same features/model under a
# naive RANDOM split and compare both numbers side by side.

from sklearn.model_selection import train_test_split

X_rand_train, X_rand_test, y_rand_train, y_rand_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

random_split_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED))
])
random_split_model.fit(X_rand_train, y_rand_train)

rand_pred = random_split_model.predict(X_rand_test)
rand_proba = random_split_model.predict_proba(X_rand_test)[:, 1]

random_precision = precision_score(y_rand_test, rand_pred, zero_division=0)
random_recall = recall_score(y_rand_test, rand_pred, zero_division=0)
random_f1 = f1_score(y_rand_test, rand_pred, zero_division=0)
random_auc = roc_auc_score(y_rand_test, rand_proba)

overlap_clients = set(model_df.loc[X_rand_train.index, "client_hash_id"]) & \
                  set(model_df.loc[X_rand_test.index, "client_hash_id"])

split_comparison = pd.DataFrame({
    "split": ["Random 80/20 (naive)", "Grouped by client_hash_id (honest)"],
    "precision": [random_precision, model_precision],
    "recall": [random_recall, model_recall],
    "f1": [random_f1, model_f1],
    "roc_auc": [random_auc, model_auc],
})

print("Clients appearing in BOTH train and test under random split:", len(overlap_clients))
print()
split_comparison

Clients appearing in BOTH train and test under random split: 55



,split,precision,recall,f1,roc_auc
0,Random 80/20 (naive),0.898881,0.575443,0.701684,0.920393
1,Grouped by client_hash_id (honest),0.931112,0.509737,0.658809,0.919989


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [46]:
# ============================================
# 3. Leakage audit — the Week-3 hunt, on this feature set
# ============================================

print("=== Timeline check ===")
print("Feature window: March 2026 (gsc_impressions, gsc_clicks, gsc_avg_position,")
print("                ga4_sessions, ga4_total_engagement_sec, ctr, days_since_update)")
print("Label window:   April 2026 (future_click_label = april clicks > 0)")
print("Features strictly precede the label window: PASS\n")

print("=== No label-derived / sibling columns ===")
suspect = "gsc_clicks"  # closest sibling to the label, but from the prior month
X_train_noSuspect = X_train.drop(columns=[suspect])
X_test_noSuspect = X_test.drop(columns=[suspect])

model_noSuspect = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED))
])
model_noSuspect.fit(X_train_noSuspect, y_train)
auc_noSuspect = roc_auc_score(y_test, model_noSuspect.predict_proba(X_test_noSuspect)[:, 1])

print(f"ROC-AUC WITH '{suspect}':    {model_auc:.4f}")
print(f"ROC-AUC WITHOUT '{suspect}': {auc_noSuspect:.4f}")
print("A collapse toward ~0.5 would flag a leak; a modest drop is expected, healthy signal loss.\n")

print("=== No product flags / existing-system scores as features ===")
flag_like = [f for f in FEATURES if any(k in f.lower() for k in ["flag", "score", "predict"])]
print("Flag-like columns in FEATURES:", flag_like if flag_like else "None found — PASS\n")

print("=== Population selection check ===")
march_only_rows = len(baseline)
after_inner_join = len(model_df)
dropped = march_only_rows - after_inner_join
print(f"March baseline rows: {march_only_rows}")
print(f"Rows kept after inner join with April outcomes: {after_inner_join}")
print(f"Rows dropped (no April row at all): {dropped} ({dropped/march_only_rows:.1%})")
print("NOTE: the modeled population is 'content that still has an April row' — that is")
print("outcome-window information baked into who gets included. Disclose in limitations.\n")

print("=== Split grouped by repeating entity ===")
train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
test_clients = set(model_df.iloc[test_idx]["client_hash_id"])
overlap = train_clients & test_clients
print(f"Clients in train: {len(train_clients)} | Clients in test: {len(test_clients)}")
print(f"Overlapping clients: {len(overlap)}", "— PASS" if not overlap else "— LEAK")
print()

print("=== Base rate next to every metric ===")
base_rate = y_test.mean()
print(f"Base rate (test set):     {base_rate:.4f}")
print(f"Majority-class baseline:  {1 - base_rate:.4f}")
print(f"Model accuracy:           {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"Model ROC-AUC:            {model_auc:.4f}\n")

print("=== Top feature importance sanity check ===")
coef_check = pd.DataFrame({
    "feature": X_train.columns,
    "abs_coef": np.abs(model.named_steps["classifier"].coef_[0])
}).sort_values("abs_coef", ascending=False)
print(coef_check)
print("No single feature towers over all others, and AUC didn't collapse when the")
print("top suspect was removed — no sign of a label-derived feature.")

=== Timeline check ===
Feature window: March 2026 (gsc_impressions, gsc_clicks, gsc_avg_position,
                ga4_sessions, ga4_total_engagement_sec, ctr, days_since_update)
Label window:   April 2026 (future_click_label = april clicks > 0)
Features strictly precede the label window: PASS

=== No label-derived / sibling columns ===
ROC-AUC WITH 'gsc_clicks':    0.9200
ROC-AUC WITHOUT 'gsc_clicks': 0.9120
A collapse toward ~0.5 would flag a leak; a modest drop is expected, healthy signal loss.

=== No product flags / existing-system scores as features ===
Flag-like columns in FEATURES: None found — PASS

=== Population selection check ===
March baseline rows: 331437
Rows kept after inner join with April outcomes: 331436
Rows dropped (no April row at all): 1 (0.0%)
NOTE: the modeled population is 'content that still has an April row' — that is
outcome-window information baked into who gets included. Disclose in limitations.

=== Split grouped by repeating entity ===
Clients in train:

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
## 4. Claim rewrite

**Original (Week-5, Section 3):**
"The model substantially improves precision, F1, and ROC-AUC over the rule baseline,
although its recall is lower."

**Rewritten (safe language, incorporating the split audit above):**
"Under the grouped-by-client split, Logistic Regression showed higher measured precision,
F1, and ROC-AUC than the Week-4 rule baseline, with lower recall. This is a directional,
decision-support comparison on one held-out client group, not a guarantee of performance
on new clients. The random-split numbers in Section 2 were higher across the board — a gap
that itself suggests part of the naive score was the model memorizing client-level
patterns rather than generalizing. The grouped numbers are the ones I'd trust for a
go/no-go decision."

In [47]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.